In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, mean_absolute_error
import pickle
import warnings
warnings.filterwarnings('ignore')

master = pd.read_csv('../data/processed/master.csv')
master['order_purchase_timestamp'] = pd.to_datetime(master['order_purchase_timestamp'])

print(f"Rows loaded: {len(master):,}")
print("Setup done")

Rows loaded: 115,723
Setup done


In [2]:
# Select features for model
ml_data = master[[
    'review_score', 'delivery_days', 'payment_value',
    'customer_state', 'product_category_name',
    'payment_installments', 'payment_type'
]].dropna()

# Encode categorical columns
le_state    = LabelEncoder()
le_category = LabelEncoder()
le_payment  = LabelEncoder()

ml_data = ml_data.copy()
ml_data['state_enc']    = le_state.fit_transform(ml_data['customer_state'])
ml_data['category_enc'] = le_category.fit_transform(ml_data['product_category_name'])
ml_data['payment_enc']  = le_payment.fit_transform(ml_data['payment_type'])

# Features and target
features = ['delivery_days', 'payment_value', 'payment_installments',
            'state_enc', 'category_enc', 'payment_enc']

X = ml_data[features]
y = ml_data['review_score'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"Features         : {features}")

Training samples : 90,580
Test samples     : 22,645
Features         : ['delivery_days', 'payment_value', 'payment_installments', 'state_enc', 'category_enc', 'payment_enc']


In [3]:
# Train Random Forest Classifier
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
accuracy = (y_pred == y_test).mean()
print(f"Review Score Predictor Accuracy: {accuracy:.2%}")
print()
print(classification_report(y_test, y_pred))

Review Score Predictor Accuracy: 60.21%

              precision    recall  f1-score   support

           1       0.57      0.27      0.37      2612
           2       1.00      0.01      0.03       770
           3       1.00      0.00      0.01      1865
           4       0.82      0.00      0.01      4324
           5       0.60      0.99      0.75     13074

    accuracy                           0.60     22645
   macro avg       0.80      0.26      0.23     22645
weighted avg       0.69      0.60      0.48     22645



In [4]:
# Delivery delay = took more than 15 days (1 = delayed, 0 = on time)
ml_data['is_delayed'] = (ml_data['delivery_days'] > 15).astype(int)

X_delay = ml_data[['payment_value', 'payment_installments',
                    'state_enc', 'category_enc', 'payment_enc']]
y_delay = ml_data['is_delayed']

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_delay, y_delay, test_size=0.2, random_state=42)

delay_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
delay_clf.fit(Xd_train, yd_train)

delay_acc = (delay_clf.predict(Xd_test) == yd_test).mean()
print(f"Delivery Delay Predictor Accuracy: {delay_acc:.2%}")

Delivery Delay Predictor Accuracy: 77.17%


In [5]:
# RFM = Recency, Frequency, Monetary
snapshot_date = master['order_purchase_timestamp'].max()

rfm = master.groupby('customer_id').agg(
    recency  =('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary =('payment_value', 'sum')
).reset_index()

# Score each dimension 1-4
rfm['r_score'] = pd.qcut(rfm['recency'],  q=4, labels=[4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], q=4, labels=[1,2,3,4]).astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

# Segment based on score
def segment(score):
    if score >= 10: return 'Champion'
    elif score >= 8: return 'Loyal'
    elif score >= 6: return 'Potential'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

rfm['segment'] = rfm['rfm_score'].apply(segment)

seg_summary = rfm.groupby('segment').agg(
    customers=('customer_id', 'count'),
    avg_monetary=('monetary', 'mean'),
    avg_recency=('recency', 'mean')
).reset_index().round(2)

print(seg_summary.to_string(index=False))
rfm.to_csv('../data/processed/rfm_segments.csv', index=False)
seg_summary.to_csv('../data/processed/rfm_summary.csv', index=False)
print("Saved RFM files")

  segment  customers  avg_monetary  avg_recency
  At Risk      13656         68.05       362.32
 Champion      15242        384.96       113.32
     Lost       1535         42.96       454.00
    Loyal      33063        243.69       196.90
Potential      32982        150.44       278.58
Saved RFM files


In [6]:
# Composite seller score
seller_score = master.groupby('seller_id').agg(
    total_revenue  =('payment_value', 'sum'),
    total_orders   =('order_id', 'nunique'),
    avg_review     =('review_score', 'mean'),
    avg_delivery   =('delivery_days', 'mean')
).reset_index()

# Only sellers with 10+ orders
seller_score = seller_score[seller_score['total_orders'] >= 10].copy()

# Normalise each metric 0-100
def normalise(series, reverse=False):
    mn, mx = series.min(), series.max()
    norm = (series - mn) / (mx - mn) * 100
    return 100 - norm if reverse else norm

seller_score['revenue_score']  = normalise(seller_score['total_revenue'])
seller_score['review_score_n'] = normalise(seller_score['avg_review'])
seller_score['delivery_score'] = normalise(seller_score['avg_delivery'], reverse=True)

# Weighted composite score
seller_score['composite_score'] = (
    seller_score['revenue_score']  * 0.4 +
    seller_score['review_score_n'] * 0.4 +
    seller_score['delivery_score'] * 0.2
).round(1)

seller_score['grade'] = pd.cut(seller_score['composite_score'],
    bins=[0,40,60,80,100],
    labels=['D','C','B','A'])

seller_score = seller_score.sort_values('composite_score', ascending=False)
seller_score['seller_short'] = seller_score['seller_id'].str[:8]

seller_score.to_csv('../data/processed/seller_scorecard.csv', index=False)
print(f"Sellers scored: {len(seller_score):,}")
print(seller_score[['seller_short','composite_score','grade','avg_review','avg_delivery']].head(10).to_string(index=False))

Sellers scored: 1,238
seller_short  composite_score grade  avg_review  avg_delivery
    7c67e144             68.1     B    3.400692     22.041265
    da8622b1             65.2     B    4.075152     10.646739
    1025f0e2             64.9     B    3.879808     11.519074
    53243585             64.8     B    4.120000     12.850117
    1f50f920             62.6     B    3.988018     15.173048
    4869f7a5             62.3     B    4.123077     14.744482
    fa1c13f2             62.2     B    4.376254     12.827243
    955fee92             62.2     B    4.091823     10.294079
    4a3ca931             62.0     B    3.824654     13.800095
    7e93a43e             60.9     B    4.363363     10.874251


In [7]:
import os

models_dir = os.path.join(os.path.dirname(os.getcwd()), 'dashboard', 'models')
os.makedirs(models_dir, exist_ok=True)

# Save models
with open(os.path.join(models_dir, 'review_predictor.pkl'), 'wb') as f:
    pickle.dump(clf, f)

with open(os.path.join(models_dir, 'delay_predictor.pkl'), 'wb') as f:
    pickle.dump(delay_clf, f)

# Save encoders
with open(os.path.join(models_dir, 'encoders.pkl'), 'wb') as f:
    pickle.dump({
        'state': le_state,
        'category': le_category,
        'payment': le_payment
    }, f)

# Save encoder labels for dashboard dropdowns
pd.DataFrame({'state': le_state.classes_}).to_csv(
    os.path.join(models_dir, 'states.csv'), index=False)
pd.DataFrame({'category': le_category.classes_}).to_csv(
    os.path.join(models_dir, 'categories.csv'), index=False)
pd.DataFrame({'payment': le_payment.classes_}).to_csv(
    os.path.join(models_dir, 'payments.csv'), index=False)

print("All models and encoders saved to dashboard/models/")
print(os.listdir(models_dir))

All models and encoders saved to dashboard/models/
['categories.csv', 'delay_predictor.pkl', 'encoders.pkl', 'payments.csv', 'review_predictor.pkl', 'states.csv']


In [8]:
import shutil

src = '../data/processed/'
dst = '../dashboard/data/'

new_files = ['rfm_segments.csv', 'rfm_summary.csv', 'seller_scorecard.csv']

for f in new_files:
    shutil.copy(src + f, dst + f)
    print(f"Copied {f}")

print("Done — dashboard/data is up to date")

Copied rfm_segments.csv
Copied rfm_summary.csv
Copied seller_scorecard.csv
Done — dashboard/data is up to date
